# 06 — Probability Calibration

Picks up from notebook 05's final model (AUC-ROC 0.751, AUC-PR 0.532,
threshold 0.51, cost ₹641,050). That model ranks orders well, but the raw
XGBoost output isn't necessarily a trustworthy probability — a predicted 0.7
doesn't have to mean 70% of orders scored that way are actually bad. This
notebook tests whether isotonic calibration is worth adding before this
reaches the app.

Assumes `model_xgb`, `x_train`, `y_train`, `x_test`, `y_test`,
`num_features_v2`, `cat_features` from notebook 05.


## Method

Calibrating on data the model already trained on would just confirm it's
confident about things it memorized, so the calibrator needs its own
held-out slice — carved out of `x_train`, with `x_test` staying completely
untouched throughout. `CalibratedClassifierCV` with `FrozenEstimator` is
used so the XGBoost model itself is never refit, only the probability
mapping on top of it.

Two calibration-slice sizes are tried, to check whether a smaller slice
(more data left for the model) closes the gap.


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

FEATURES = num_features_v2 + cat_features
preprocessor = model_xgb.named_steps["prep"]
cost_fp, cost_fn = 50, 300

def transform(df):
    arr = preprocessor.transform(df)
    return arr.toarray() if hasattr(arr, "toarray") else arr

X_test_transformed = transform(x_test[FEATURES])

def calibration_trial(calib_fraction, random_state=42):
    x_fit, x_calib, y_fit, y_calib = train_test_split(
        x_train[FEATURES], y_train, test_size=calib_fraction,
        stratify=y_train, random_state=random_state,
    )

    fit_preprocessor = preprocessor  # already fit on the full x_train in notebook 05
    X_fit_t = transform(x_fit)
    X_calib_t = transform(x_calib)

    scale_pos_weight = (y_fit == 0).sum() / (y_fit == 1).sum()
    xgb_trial = xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight, eval_metric="aucpr", random_state=42,
    )
    xgb_trial.fit(X_fit_t, y_fit)

    calibrated = CalibratedClassifierCV(FrozenEstimator(xgb_trial), method="isotonic")
    calibrated.fit(X_calib_t, y_calib)

    raw_probs = xgb_trial.predict_proba(X_test_transformed)[:, 1]
    cal_probs = calibrated.predict_proba(X_test_transformed)[:, 1]

    best_t, best_cost = None, float("inf")
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (cal_probs > t).astype(int)
        fp = ((preds == 1) & (y_test == 0)).sum()
        fn = ((preds == 0) & (y_test == 1)).sum()
        c = fp * cost_fp + fn * cost_fn
        if c < best_cost:
            best_cost, best_t = c, t

    preds_at_best = (cal_probs > best_t).astype(int)
    tp = ((preds_at_best == 1) & (y_test == 1)).sum()
    fp = ((preds_at_best == 1) & (y_test == 0)).sum()
    fn = ((preds_at_best == 0) & (y_test == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    return {
        "calib_fraction": calib_fraction,
        "auc_roc": roc_auc_score(y_test, cal_probs),
        "auc_pr": average_precision_score(y_test, cal_probs),
        "brier_raw": brier_score_loss(y_test, raw_probs),
        "brier_calibrated": brier_score_loss(y_test, cal_probs),
        "best_threshold": best_t,
        "cost": best_cost,
        "precision": precision,
        "recall": recall,
        "raw_probs": raw_probs,
        "cal_probs": cal_probs,
    }

trial_25 = calibration_trial(0.25)
trial_10 = calibration_trial(0.10)


In [ ]:
summary = pd.DataFrame([
    {"model": "Original (no calibration, 80% train)", "auc_roc": 0.751, "auc_pr": 0.532,
     "precision": 0.37, "recall": 0.54, "cost": 641_050},
    {"model": "Calibrated, 75/25 fit/calib split", "auc_roc": trial_25["auc_roc"], "auc_pr": trial_25["auc_pr"],
     "precision": trial_25["precision"], "recall": trial_25["recall"], "cost": trial_25["cost"]},
    {"model": "Calibrated, 90/10 fit/calib split", "auc_roc": trial_10["auc_roc"], "auc_pr": trial_10["auc_pr"],
     "precision": trial_10["precision"], "recall": trial_10["recall"], "cost": trial_10["cost"]},
]).set_index("model")
summary


**Result** (from the actual runs behind this notebook):

| Model | AUC-ROC | AUC-PR | Precision | Recall | Cost |
|---|---|---|---|---|---|
| Original (no calibration, 80% train) | 0.751 | 0.532 | 0.37 | 0.54 | ₹641,050 |
| Calibrated, 75/25 fit/calib split | 0.719 | 0.417 | 0.30 | 0.54 | ₹697,550 |
| Calibrated, 90/10 fit/calib split | 0.725 | 0.417 | 0.33 | 0.51 | ₹689,250 |

Brier score (lower is better) in both trials: **0.172 uncalibrated → 0.112
calibrated** — the isotonic mapping is doing exactly what it's supposed to,
the probabilities really are more trustworthy after calibration.

But AUC-PR barely moved between the two trials (0.417 in both, despite the
90/10 split leaving the model with noticeably more training data than
75/25) — if this were simply a data-quantity problem, giving the model 20%
more rows should have closed more of the gap than it did. That points to
something more structural than "not enough training data": isotonic
regression is a step function, and with a calibration slice this size it
likely coarsens the score distribution, blunting the fine-grained ranking
the app's 0.01-step threshold sweep depends on — consistent with the
optimal threshold collapsing from 0.51 to 0.11 in both trials, a much
bigger shift than calibration alone should cause.


## Reliability curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for ax, trial, label in [(axes[0], trial_25, "75/25 split"), (axes[1], trial_10, "90/10 split")]:
    frac_raw, mean_raw = calibration_curve(y_test, trial["raw_probs"], n_bins=10, strategy="quantile")
    frac_cal, mean_cal = calibration_curve(y_test, trial["cal_probs"], n_bins=10, strategy="quantile")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
    ax.plot(mean_raw, frac_raw, marker="o", label="Uncalibrated")
    ax.plot(mean_cal, frac_cal, marker="o", label="Isotonic-calibrated")
    ax.set_title(label)
    ax.set_xlabel("Mean predicted probability")

axes[0].set_ylabel("Fraction of actual positives")
axes[0].legend()
plt.tight_layout()
plt.savefig("../docs/calibration_curve.png", dpi=150)
plt.show()


## Decision

Calibration is **not adopted** for the deployed model. The metric the app
is actually built around — total cost at the chosen threshold — got worse
in both attempts (₹697,550 and ₹689,250 vs. ₹641,050 for the original), and
that's the number that matters here, not the Brier score improvement on its
own.

This isn't a case against probability calibration in general — it's a
genuinely useful technique when a system needs to reason about probability
values directly (e.g. computing expected rupee loss per order rather than
just thresholding once). But this pipeline only uses the score to make a
single flag/no-flag decision at one cost-optimal cutoff, and for that use
case, ranking quality matters more than calibrated probability values —
which is exactly what got traded away here.

The deployed model stays the original XGBoost pipeline trained on the full
80% training split (`src/train_final_model.py`), with no calibration step.
This notebook is kept as the record of testing the idea and the reasoning
for not shipping it.
